# Modulo 2 · Lezione 3 — GitHub e repository remoti

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vediamo oggi?

| # | Blocco | Cosa imparerai |
|---|--------|----------------|
| 1 | Git vs GitHub | La differenza fondamentale |
| 2 | Creare account e primo repo su GitHub | Configurazione iniziale della piattaforma |
| 3 | SSH vs HTTPS | Come autenticarti in modo sicuro |
| 4 | `git remote` | Collegare il tuo repo locale con il remoto |
| 5 | `git push` | Caricare le modifiche sul remoto |
| 6 | `git clone` | Scaricare un repo esistente |
| 7 | `git fetch` e `git pull` | Ricevere le modifiche dal remoto |
| 8 | Fork | Contribuire a progetti altrui |
| 9 | Pratica | Pubblicare il sistema di note su GitHub |

> 💡 **Idea centrale:** Git è lo strumento locale. GitHub è la piattaforma dove il lavoro viene condiviso, revisionato e su cui si collabora. Oggi impariamo a collegare i due mondi.

---
## 1. Git vs GitHub — Non sono la stessa cosa

```
GIT                                GITHUB
───────────────────────────────    ──────────────────────────────────
Strumento a riga di comando        Piattaforma web (azienda di Microsoft)
Gira sulla tua macchina            Gira sui server di GitHub
Gestisce la cronologia locale      Ospita repository remoti
Creato da Linus Torvalds (2005)    Fondato nel 2008
Open source                        Servizio freemium
Senza di lui, GitHub non funziona  Senza di lui, Git continua a funzionare da solo
```

**Alternative a GitHub** (tutte usano Git):

| Piattaforma | Quando si usa |
|------------|---------------|
| **GitLab** | Aziende che vogliono self-hosting o CI/CD integrato |
| **Bitbucket** | Team che usano l'ecosistema Atlassian (Jira, Confluence) |
| **Azure DevOps** | Ambienti Microsoft / .NET |
| **Gitea** | Self-hosted, molto leggero |

---
## 2. Creare account e primo repository su GitHub

### Passo dopo passo (da fare in classe)

1. Vai su **github.com** → Sign up
2. Scegli un nome utente (farà parte delle tue URL: `github.com/tuo-utente`)
3. Conferma l'email
4. Crea un nuovo repository:
   - **Repository name:** `sistema-notas`
   - **Description:** Sistema di note da riga di comando
   - **Visibility:** Public (così tutti possono vederlo)
   - ⬛ **Non** inizializzarlo con README — ne abbiamo già uno locale
   - Clic su **Create repository**

### Cosa vedi quando crei un repo vuoto

GitHub ti mostra esattamente i comandi per collegare il tuo repo locale. È la pagina più utile di tutta la piattaforma:

```bash
# Se hai già il repo locale (il nostro caso):
git remote add origin https://github.com/TUO-UTENTE/sistema-notas.git
git branch -M main
git push -u origin main
```

---
## 3. SSH vs HTTPS — Come autenticarti

Perché Git possa caricare modifiche su GitHub, deve identificarti. Ci sono due metodi:

```
HTTPS                              SSH
─────────────────────────────────  ──────────────────────────────────
https://github.com/user/repo.git   git@github.com:user/repo.git
Chiede utente/token ogni volta*    Autenticazione automatica con chiave
Più facile da configurare          Più sicuro e comodo a lungo termine
Funziona su reti restrittive       Può essere bloccato su alcune reti

* Con il credential manager viene salvato.
```

**Consiglio:** configura SSH — lo fai una volta sola e dimentichi le password per sempre.

### Configurare SSH su GitHub (una sola volta)

In [ ]:
# 1. Verificare se hai già chiavi SSH
!ls ~/.ssh/*.pub 2>/dev/null && echo "Hai già delle chiavi SSH" || echo "Non hai ancora chiavi SSH"

In [ ]:
# 2. Generare una nuova chiave SSH (solo se non ne hai)
# Sostituisci l'email con quella del tuo account GitHub
#
# !ssh-keygen -t ed25519 -C "tu@email.com" -f ~/.ssh/id_ed25519 -N ""
#
# ed25519 è l'algoritmo moderno consigliato
# -N "" crea la chiave senza passphrase (più comodo per l'uso quotidiano)

print("Per generare la chiave SSH, esegui nel terminale:")
print('ssh-keygen -t ed25519 -C "tu@email.com"')
print()
print("Poi mostra la chiave pubblica con:")
!cat ~/.ssh/id_ed25519.pub 2>/dev/null || echo "(genera prima la chiave)"

### Aggiungere la chiave pubblica a GitHub

1. Copia il contenuto di `~/.ssh/id_ed25519.pub` (tutta la riga)
2. Vai su **GitHub → Settings → SSH and GPG keys → New SSH key**
3. Incolla la chiave e dalle un nome descrittivo (es: `Portatile MacBook`)
4. Clic su **Add SSH key**

In [ ]:
# Verificare che la connessione SSH con GitHub funzioni
!ssh -T git@github.com 2>&1 || true
# Risposta attesa: "Hi TUO-UTENTE! You've successfully authenticated..."

---
## 4. `git remote` — Collegare locale e remoto

Un **remote** è un alias che punta a un URL di repository. Il nome convenzionale per il remoto principale è **`origin`**.

```bash
git remote add <nome> <url>      # aggiungere un remoto
git remote -v                    # vedere i remoti configurati
git remote remove <nome>         # eliminare un remoto
git remote rename <old> <new>    # rinominare
git remote set-url <nome> <url>  # cambiare l'URL
```

In [ ]:
import os
from pathlib import Path

# Posizioniamoci nel repo sistema_notas della lezione precedente
NOTAS = Path.home() / "sistema_notas"
if NOTAS.exists():
    os.chdir(NOTAS)
    print(f"Repo: {Path.cwd()}")
    !git log --oneline
else:
    print("⚠️  Crea prima il repo sistema_notas (Lezione 2, Esercizio 1)")

In [ ]:
# Collegarsi a GitHub
# CAMBIA questo URL con quello del tuo repository su GitHub
GITHUB_URL = "git@github.com:TUO-UTENTE/sistema-notas.git"   # SSH
# GITHUB_URL = "https://github.com/TUO-UTENTE/sistema-notas.git"  # HTTPS

# !git remote add origin {GITHUB_URL}

# Vedere i remoti configurati
!git remote -v

---
## 5. `git push` — Caricare le modifiche sul remoto

```bash
git push -u origin main    # primo push: stabilisce il tracking
git push                   # push successivi (già sa dove andare)
git push origin main       # push esplicito di un branch
git push --tags            # caricare anche i tag
```

**Cosa fa `-u`?**
`-u` (o `--set-upstream`) dice a Git che il branch locale `main` fa tracking del branch remoto `origin/main`. Dopo il primo push con `-u`, puoi usare semplicemente `git push` e `git pull`.

```
LOCALE                             REMOTO (GitHub)
──────                             ──────────────────
main ────── git push ──────────►  origin/main
main ◄───── git pull ──────────── origin/main
```

In [ ]:
# Assicurarci di essere su main
!git branch -M main

# Primo push (decommenta quando hai configurato il remote)
# !git push -u origin main

# Verificare lo stato del tracking
!git branch -vv

In [ ]:
# Caricare anche i tag (i tag non vengono caricati con git push normale)
# !git push --tags

# Oppure caricare un tag specifico
# !git push origin v1.0

print("Per caricare i tag sul remoto:")
print("  git push --tags         # tutti i tag")
print("  git push origin v1.0   # un tag specifico")

---
## 6. `git clone` — Scaricare un repo esistente

`git clone` scarica un repository completo — con tutta la sua cronologia — e configura automaticamente il remote `origin`.

```bash
git clone <url>                  # clona in una cartella con il nome del repo
git clone <url> nome-locale      # clona con un nome di cartella personalizzato
git clone <url> --depth 1        # shallow clone: solo l'ultimo commit (più veloce)
```

**Differenza tra `clone` e `init + remote add`:**

| | `git clone` | `git init` + `git remote add` |
|-|-------------|--------------------------------|
| Quando | Il repo esiste GIÀ su GitHub | Hai codice locale che vuoi caricare |
| Cosa fa | Scarica tutto + configura origin | Crei il repo da zero |
| Cronologia | La scarica completa | Inizia vuota |

In [ ]:
import os, shutil
from pathlib import Path

# Clonare un repo pubblico reale (senza bisogno di autenticazione)
DESTINO = Path.home() / "clone_demo"
if DESTINO.exists():
    shutil.rmtree(DESTINO)

os.chdir(Path.home())

# Cloniamo un repo piccolo e pubblico come esempio
!git clone https://github.com/octocat/Hello-World clone_demo

os.chdir(DESTINO)
print("\nContenuto del repo clonato:")
!ls -la
print("\nCronologia:")
!git log --oneline
print("\nRemote configurato automaticamente:")
!git remote -v

---
## 7. `git fetch` e `git pull` — Ricevere le modifiche

Quando qualcuno (o tu da un altro computer) fa un push sul remoto, devi portare quelle modifiche sulla tua macchina.

```
git fetch                         git pull
──────────────────────────────    ──────────────────────────────────
Scarica le modifiche dal remoto   Scarica E fonde nel tuo branch attuale
NON tocca la tua working directory SÌ modifica la tua working directory
Sicuro, puoi sempre farlo          Può creare conflitti
Devi fare il merge dopo            fetch + merge in un solo passaggio

git pull = git fetch + git merge
```

**Quale usare?**
- `git fetch` → quando vuoi vedere cosa c'è nel remoto prima di integrare.
- `git pull` → quando sei sicuro che non ci saranno conflitti (lavoro da solo).
- Nei team: `git fetch` + revisione + `git merge` o `git rebase`.

In [ ]:
os.chdir(DESTINO)

# fetch scarica le informazioni senza toccare i tuoi file
!git fetch origin
print("\nDopo git fetch:")
!git log --oneline --all   # --all mostra anche i branch remoti

# Vedere la differenza tra locale e remoto
print("\nDifferenze tra locale e origin/main:")
!git log HEAD..origin/main --oneline 2>/dev/null || echo "Sei aggiornato"

In [ ]:
# pull = fetch + merge
# (In questo repo demo non ci saranno modifiche nuove, ma il comando è lo stesso)
!git pull origin main

print("\nStato dopo pull:")
!git status

### Il flusso quando il remoto ha modifiche nuove

```
SITUAZIONE TIPICA IN UN TEAM:

Remoto:  A ── B ── C ── D   (il tuo collega ha fatto C e D)
Locale:  A ── B ── E         (tu hai fatto E senza sapere di C e D)

git fetch:    Scarica C e D, li conosce ma non li fonde
git merge:    A ── B ── C ── D ── E ── M  (M = merge commit)
git rebase:   A ── B ── C ── D ── E'      (cronologia lineare, più pulita)

git pull --rebase  →  fa fetch + rebase automaticamente
```

---
## 8. Fork — Contribuire a progetti altrui

Un **fork** è una copia personale di un repository altrui sul tuo account GitHub. Ti permette di proporre modifiche a progetti dove non hai permessi di scrittura.

```
FLUSSO DI CONTRIBUZIONE CON FORK:

1. Fork del repo originale ─────►  tuo-utente/progetto (su GitHub)
                                          │
2. git clone del tuo fork ────────────────►  la tua macchina
                                          │
3. Creare branch: git switch -c mio-miglioramento │
                                          │
4. Fare modifiche + commit                │
                                          │
5. git push origin mio-miglioramento ─────►  il tuo fork su GitHub
                                          │
6. Aprire Pull Request ──────────────────► progetto-originale/main
                                          │
7. Il maintainer revisiona e accetta (merge) │
```

**Fork vs Clone:**

| | Clone | Fork |
|-|-------|------|
| Dove | Solo sulla tua macchina | Su GitHub + la tua macchina |
| Puoi fare push | Solo se hai i permessi | Sempre (è tuo) |
| Per contribuire | No | Sì |

In [ ]:
# Quando hai un fork, è buona pratica mantenere sincronizzato
# il remoto del progetto originale (chiamato 'upstream' per convenzione)

print("Flusso per mantenere un fork sincronizzato:")
flujo = """
# Aggiungere il repo originale come 'upstream'
git remote add upstream https://github.com/ORIGINALE/progetto.git

# Vedere tutti i remoti
git remote -v
# origin    git@github.com:TUO-UTENTE/progetto.git  (il tuo fork)
# upstream  https://github.com/ORIGINALE/progetto.git (l'originale)

# Sincronizzare il tuo fork con l'originale
git fetch upstream
git merge upstream/main
git push origin main
"""
print(flujo)

---
## 9. Pratica — Pubblicare il sistema di note su GitHub

In questa pratica pubblichiamo il repo `sistema_notas` della Lezione 2 su GitHub e ci esercitiamo con il flusso completo push/pull.

> ⚠️ Per gli esercizi 2–5 hai bisogno di un remote configurato su GitHub. L'esercizio 1 funziona senza account.

---

### Esercizio 1 — Preparare il repo per la pubblicazione

**Consegna:** Prima di caricare su GitHub, un repo professionale ha bisogno di alcuni file. Aggiungili al repo `sistema_notas`:

1. **`README.md` completo** con: titolo, descrizione, sezione di installazione, sezione d'uso e sezione sulla struttura del progetto.
2. **`LICENSE`** con la licenza MIT (cercala su choosealicense.com e incollala).
3. **`requirements.txt`** con le dipendenze del progetto (in questo caso nessuna, ma il file deve esistere vuoto o con un commento).
4. Committa i tre file con il messaggio appropriato.
5. Verifica la cronologia finale.

In [ ]:
# ✏️ IL TUO CODICE QUI
import os
from pathlib import Path

NOTAS = Path.home() / "sistema_notas"
os.chdir(NOTAS)
print(f"Lavorando in: {Path.cwd()}")


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
from datetime import date

# README.md completo
Path("README.md").write_text(f"""# Sistema di Note

Applicazione da riga di comando per gestire note personali.
Progetto di pratica del Corso di IA.

## Installazione

```bash
git clone https://github.com/TUO-UTENTE/sistema-notas.git
cd sistema-notas
```

## Uso

```python
from gestor import GestorNotas

g = GestorNotas()
g.crear('Riunione', 'Preparare la presentazione di lunedì')
g.listar()
resultados = g.buscar('riunione')
g.eliminar(1)
```

## Struttura

```
sistema_notas/
├── nota.py          # Modello dei dati
├── gestor.py        # Logica di business
├── test_gestor.py   # Test
└── README.md        # Questo file
```

## Licenza

MIT © {date.today().year}
""", encoding="utf-8")

# LICENSE — MIT
Path("LICENSE").write_text(f"""MIT License

Copyright (c) {date.today().year} Curso IA

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
""", encoding="utf-8")

# requirements.txt
Path("requirements.txt").write_text(
    "# Questo progetto non ha dipendenze esterne\n",
    encoding="utf-8"
)

!git add README.md LICENSE requirements.txt
!git commit -m "docs: aggiungere README completo, licenza e requirements"

print("\nCronologia finale:")
!git log --oneline
```

**Perché questi file:**
- `README.md` è la prima cosa che vede chiunque su GitHub — è la copertina del progetto.
- `LICENSE` indica cosa possono fare gli altri con il tuo codice. Senza licenza, il codice è tecnicamente a uso ristretto.
- `requirements.txt` è lo standard Python per dichiarare le dipendenze — `pip install -r requirements.txt` installa tutto.
</details>

---
### Esercizio 2 — Pubblicare su GitHub

**Consegna:** Carica il repo `sistema_notas` su GitHub:

1. Crea il repository su GitHub (senza inizializzarlo con README).
2. Aggiungi il remote `origin` che punta al tuo repository.
3. Verifica con `git remote -v` che punti al posto giusto.
4. Fai il primo push con `-u` per stabilire il tracking.
5. Verifica nel browser che i file siano su GitHub.
6. Mostra lo stato del tracking con `git branch -vv`.

In [ ]:
# ✏️ IL TUO CODICE QUI
# Sostituisci TUO-UTENTE con il tuo nome utente su GitHub

# !git remote add origin git@github.com:TUO-UTENTE/sistema-notas.git
# !git remote -v
# !git branch -M main
# !git push -u origin main
# !git branch -vv


---
### Esercizio 3 — Il ciclo push/pull

**Consegna:** Simula il flusso di lavoro quotidiano:

1. Modifica `gestor.py` localmente: aggiungi un metodo `contar()` che restituisca il numero di note.
2. Committa la modifica localmente.
3. Carica la modifica su GitHub con `git push`.
4. Verifica nel browser che la modifica appaia su GitHub.
5. Ora simula di essere su «un altro computer»: clona il tuo stesso repo in una directory diversa (`sistema_notas_copia`).
6. Verifica che `gestor.py` nella copia abbia il metodo `contar()`.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
import os, shutil
from pathlib import Path

# Assicurarci di essere in sistema_notas
os.chdir(Path.home() / "sistema_notas")

# 1 e 2. Aggiungere il metodo e committare
gestor = Path("gestor.py").read_text(encoding="utf-8")
gestor += """
    def contar(self):
        return len(self._notas)
"""
Path("gestor.py").write_text(gestor, encoding="utf-8")

!git add gestor.py
!git commit -m "feat: aggiungere il metodo contar in GestorNotas"

# 3. Caricare su GitHub
# !git push

# 4. Verificare nel browser (manuale)
print("Verifica su: https://github.com/TUO-UTENTE/sistema-notas")

# 5. Simulare "un altro computer" clonando in una directory diversa
COPIA = Path.home() / "sistema_notas_copia"
if COPIA.exists():
    shutil.rmtree(COPIA)

os.chdir(Path.home())
# !git clone git@github.com:TUO-UTENTE/sistema-notas.git sistema_notas_copia

# 6. Verificare che il metodo contar sia nella copia
# os.chdir(COPIA)
# print(Path("gestor.py").read_text())

print("\nIl flusso completo:")
print("  Locale: modificare → add → commit → push")
print("  Un altro computer: clone (o pull)")
```
</details>

---
### Esercizio 4 — Esplorare un repository open source

**Consegna:** Senza fare nulla di distruttivo, esplora un repository pubblico reale:

1. Clona `https://github.com/pallets/flask` (il framework web Flask) con `--depth 1` (shallow clone per renderlo veloce).
2. Rispondi usando solo comandi Git:
   - Quanti file ci sono nella directory radice?
   - Qual è il messaggio del commit più recente?
   - Quanti remoti ha configurati?
   - Quali branch esistono nel remoto?
3. Elimina la directory clonata quando hai finito.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
import os, shutil
from pathlib import Path

FLASK = Path.home() / "flask_demo"
if FLASK.exists(): shutil.rmtree(FLASK)
os.chdir(Path.home())

# 1. Shallow clone (solo l'ultimo commit — molto più veloce)
!git clone https://github.com/pallets/flask flask_demo --depth 1 -q
os.chdir(FLASK)

# 2a. File nella directory radice
print("=== File nella radice ===")
!ls

# 2b. Ultimo commit
print("\n=== Ultimo commit ===")
!git log --oneline -1

# 2c. Remoti
print("\n=== Remoti ===")
!git remote -v

# 2d. Branch remoti
print("\n=== Branch remoti ===")
!git branch -r

# 3. Pulire
os.chdir(Path.home())
shutil.rmtree(FLASK)
print("\nDirectory eliminata")
```

**`--depth 1`** fa uno *shallow clone* — scarica solo l'ultimo snapshot senza la cronologia completa. Per Flask (migliaia di commit) questo fa passare da diversi minuti a pochi secondi. Utile quando vuoi solo leggere il codice, non contribuire.
</details>

---
### Esercizio 5 (sfida) — Script di pubblicazione automatica

**Consegna:** Scrivi una funzione Python `publicar_repo(ruta_local, nombre_github, usuario)` che automatizzi il processo di pubblicazione di un repository locale su GitHub:

1. Verifica che la directory sia un repo Git valido.
2. Verifica che non abbia già un remote `origin`.
3. Genera e mostra le istruzioni passo dopo passo (non esegue nulla di distruttivo — solo una guida):
   - URL SSH del remote da aggiungere
   - Comandi da eseguire in ordine
   - URL del repo su GitHub dove verificare
4. Verifica lo stato attuale del repo (branch, commit in attesa di push).

In [ ]:
# ✏️ IL TUO CODICE QUI
import subprocess
from pathlib import Path


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
import subprocess
from pathlib import Path

def git_cmd(args, cwd):
    r = subprocess.run(["git"] + args, cwd=str(cwd),
                       capture_output=True, text=True)
    return r.stdout.strip(), r.returncode

def publicar_repo(ruta_local, nombre_github, usuario):
    ruta = Path(ruta_local).expanduser().resolve()

    print(f"\n{'═'*55}")
    print(f"  GUIDA ALLA PUBBLICAZIONE — {ruta.name}")
    print(f"{'═'*55}")

    # 1. Verificare che sia un repo Git
    _, code = git_cmd(["rev-parse", "--git-dir"], ruta)
    if code != 0:
        print(f"  ❌ '{ruta}' non è un repository Git")
        print(f"     Inizializzalo prima con: git init")
        return
    print(f"  ✅ Repository Git valido")

    # 2. Verificare se ha già un remote origin
    remotos, _ = git_cmd(["remote"], ruta)
    if "origin" in remotos.splitlines():
        url, _ = git_cmd(["remote", "get-url", "origin"], ruta)
        print(f"  ⚠️  Ha già un remote 'origin': {url}")
        print(f"     Eliminalo con: git remote remove origin")
        tiene_origin = True
    else:
        print(f"  ✅ Senza remote origin — pronto per il collegamento")
        tiene_origin = False

    # 3. Stato del repo
    rama, _   = git_cmd(["branch", "--show-current"], ruta)
    commits, _ = git_cmd(["rev-list", "--count", "HEAD"], ruta)
    status, _ = git_cmd(["status", "--porcelain"], ruta)

    print(f"\n  STATO ATTUALE:")
    print(f"    Branch:      {rama or 'nessun commit ancora'}")
    print(f"    Commit:      {commits or '0'}")
    if status:
        print(f"    ⚠️  Hai modifiche non committate — fai commit prima di pubblicare")
    else:
        print(f"    ✅ Working tree pulito")

    # 4. Istruzioni
    ssh_url   = f"git@github.com:{usuario}/{nombre_github}.git"
    https_url = f"https://github.com/{usuario}/{nombre_github}"

    print(f"\n  PASSI PER PUBBLICARE:")
    print(f"    1. Creare il repo su GitHub (senza inizializzarlo):")
    print(f"       https://github.com/new")
    print(f"       Nome: {nombre_github}")
    print()
    if not tiene_origin:
        print(f"    2. Collegare il remoto:")
        print(f"       git remote add origin {ssh_url}")
        print()
    print(f"    3. Assicurare il nome del branch:")
    print(f"       git branch -M main")
    print()
    print(f"    4. Primo push:")
    print(f"       git push -u origin main")
    print()
    print(f"    5. Se hai dei tag:")
    print(f"       git push --tags")
    print()
    print(f"  📍 Risultato: {https_url}")
    print(f"{'═'*55}")


publicar_repo("~/sistema_notas", "sistema-notas", "TUO-UTENTE")
```
</details>

---
## Riepilogo della lezione

```bash
# ─── SSH ───────────────────────────────────────────────────
ssh-keygen -t ed25519 -C "email"   # generare la chiave
cat ~/.ssh/id_ed25519.pub           # mostrare la chiave pubblica
ssh -T git@github.com               # verificare la connessione

# ─── Remoti ────────────────────────────────────────────────
git remote add origin <url>         # collegarsi a GitHub
git remote -v                       # vedere i remoti
git remote set-url origin <url>     # cambiare URL

# ─── Push ──────────────────────────────────────────────────
git push -u origin main             # primo push + tracking
git push                            # push successivi
git push --tags                     # caricare i tag

# ─── Clone ─────────────────────────────────────────────────
git clone <url>                     # scaricare il repo completo
git clone <url> --depth 1           # shallow clone (veloce)

# ─── Pull / Fetch ──────────────────────────────────────────
git fetch origin                    # scaricare senza fondere
git pull                            # scaricare e fondere
git pull --rebase                   # scaricare e rebase

# ─── Informazioni ──────────────────────────────────────────
git branch -vv                      # branch con info di tracking
git branch -r                       # branch remoti
git log HEAD..origin/main           # commit in attesa di pull
```

## Cosa vediamo nella prossima lezione?

Nella **Lezione 4** lavoriamo con **branch e collaborazione** — la parte di Git che rende possibile a più sviluppatori lavorare sullo stesso progetto contemporaneamente: `branch`, `switch`, `merge`, risoluzione dei conflitti e il flusso delle Pull Request.